Intro to project

In [ ]:
import joblib
from tqdm import tqdm
import pandas as pd
import numpy as np
from itertools import product

from sklearn.preprocessing import MinMaxScaler
from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import torch
import torch.nn as nn
from scipy.special import logit, expit

from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from sklearn.calibration import calibration_curve

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)

# for reproducibility
print("pandas: "+pd.__version__)
print("numpy: "+np.__version__)
print("pymc: "+pm.__version__)
import sklearn
print("sklearn: "+sklearn.__version__)
import xgboost
print("xgboost: "+xgboost.__version__)
import lightgbm
print("lgbm: "+lightgbm.__version__)
print("torch: "+torch.__version__)

In [ ]:
features = ['distance_mm', 'angle_mm']
df=pd.read_csv('data/shot_probs_data.csv')
df = df.reset_index(drop=False)
scaler = joblib.load('minmax_scaler.pkl')

X_train = df.loc[df['is_train']==1][['distance_mm','angle_mm']].values
X_test = df.loc[df['is_train']!=1][['distance_mm','angle_mm']].values
y_train=df.loc[df['is_train']==1]['is_made'].values
y_test=df.loc[df['is_train']!=1]['is_made'].values

dunks = df.loc[(df['is_train']==1)&(df['shot_type']=='DUNK')]
print(np.mean(dunks['is_made']))

jumpers = df.loc[(df['is_train']==1)&(df['shot_type']=='JUMPER')]
print(np.mean(jumpers['is_made']))

hooks = df.loc[(df['is_train']==1)&(df['shot_type']=='HOOK')]
print(np.mean(hooks['is_made']))

layups = df.loc[(df['is_train']==1)&(df['shot_type']=='LAYUP')]
print(np.mean(layups['is_made']))

In [ ]:
def compute_scores(target, preds):
    loss = log_loss(target, preds)
    auc = roc_auc_score(target, preds)
    binary_preds = (preds >= 0.5).astype(int)
    accuracy = accuracy_score(target, binary_preds)
    return loss, auc, accuracy

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

In [ ]:
# xgboost params
xg_params = {
    'n_estimators':[100, 250, 500, 1000, 1500],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.005, 0.01, 0.1, 0.2],
    'reg_lambda':[1, 2],
    'reg_alpha': [0, 0.5],
}

fixed_params = {
    'booster': 'gbtree',
    'subsample': 0.75,
    'device':'cuda',
    'eval_metric':'logloss',
    'n_jobs':1,
    'random_state':RANDOM_SEED,
}

xgb_model = XGBClassifier(**fixed_params)
gs_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=xg_params,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1
)

In [ ]:
gs_xgb.fit(X_train, y_train)
best_xg = gs_xgb.best_estimator_
best_xg_params = gs_xgb.best_params_

In [ ]:
# lgbm params
lgbm_params = {
    'n_estimators':[100, 250, 500, 1000, 1500],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.005, 0.01, 0.1, 0.2],
    'num_leaves': [7, 15, 31],
    'reg_lambda':[1, 2, 5],
    'reg_alpha': [0, 0.5, 1],
}

fixed_params = {
    'subsample': 0.75,
    'device':'gpu',
    'eval_metric':'logloss',
    'n_jobs':1,
    'random_state':RANDOM_SEED,
}

lgb_model = LGBMClassifier(**fixed_params)
gs_lgb = GridSearchCV(
    estimator=lgb_model,
    param_grid=lgbm_params,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1
)

In [ ]:
gs_lgb.fit(X_train, y_train)
best_lg = gs_lgb.best_estimator_
best_lg_params = gs_lgb.best_params_

In [ ]:
# rf params
rf_params = {
    'n_estimators': [100, 200, 500, 1000],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 10, 50],
    'min_samples_leaf': [1, 5, 20],
    'max_features':['sqrt', None]
}
rf_model = RandomForestClassifier(n_jobs=1, random_state=RANDOM_SEED)
gs_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_params,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1
)

In [ ]:
gs_rf.fit(X_train, y_train)
best_rf = gs_rf.best_estimator_
best_rf_params = gs_rf.best_params_

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout):
        super().__init__()
        layers = []
        inputs = 2 # two for angle and distance
        for h in hidden_sizes:
            layers += [
                nn.Linear(inputs, h), # connected layers
                nn.BatchNorm1d(h), # norm weights
                nn.ReLU(), # activation func
                nn.Dropout(dropout) # rando dropout neurons for generlization
            ]
            inputs = h
        layers.append(nn.Linear(inputs, 1)) # output layer
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

early_stop = EarlyStopping(
    monitor='valid_loss',
    patience=10,
    threshold=.001,
    lower_is_better=True
)

param_grid = {
    'module__hidden_sizes': [(16, 8), (32, 16), (64, 32, 16)],
    'module__dropout': [0.2, 0.4],
    'optimizer__lr': [.001, .01, 0.03],
    'batch_size': [512, 2048],
}

net = NeuralNetClassifier(
    module=MLP,
    criterion=nn.BCEWithLogitsLoss,
    optimizer=torch.optim.Adam,
    max_epochs=200,
    callbacks=[early_stop],
    device='cuda',
)

gs_nn = GridSearchCV(
    estimator=net,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=1
)

In [ ]:
gs_nn.fit(
    X_train.astype('float32'),
    y_train.astype('int32')
)